In [25]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

In [27]:
np.random.seed(42)

n_normal = 950
n_anomalies = 50

normal_amount = np.random.normal(500, 120, n_normal)
normal_time = np.random.normal(12, 3, n_normal)

anomaly_amount = np.random.uniform(1200, 2500, n_anomalies)
anomaly_time = np.random.uniform(0, 24, n_anomalies)

amount = np.concatenate([normal_amount, anomaly_amount])
time = np.concatenate([normal_time, anomaly_time])

y = np.concatenate([
    np.zeros(n_normal, dtype=int),
    np.ones(n_anomalies, dtype=int)
])

X = pd.DataFrame({
    "transaction_amount" : amount,
    "transaction_time" : time
})

In [28]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

print("Training Rows : ", len(X_train))
print("Validation Rows : ", len(X_val))
print("Testing Rows : ", len(X_test))


Training Rows :  600
Validation Rows :  200
Testing Rows :  200


In [29]:
scaler = StandardScaler()
X_train_sclaed = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [30]:
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
contamination_values = [0.01, 0.03, 0.05, 0.08, 0.10]
results = []

for contamination in contamination_values : 
    model = IsolationForest(
        n_estimators=100,
        contamination=contamination,
        random_state=42
    )

    model.fit(X_train_sclaed)
    raw_predictions = model.predict(X_val_scaled)
    y_val_pred = (raw_predictions == -1).astype(int)

    preicision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)

    results.append({
        "contamination" : contamination,
        "precision" : preicision,
        "recall" : recall, 
        "f1 score" : f1
    })
    


In [31]:
results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))

 contamination  precision  recall  f1 score
          0.01      1.000     0.5     0.667
          0.03      1.000     0.9     0.947
          0.05      1.000     1.0     1.000
          0.08      0.556     1.0     0.714
          0.10      0.476     1.0     0.645


In [32]:
best_row = results_df.loc[results_df['f1 score'].idxmax()]
best_contamination = best_row['contamination']

print("Selected Contamination : ", best_contamination)

Selected Contamination :  0.05


In [33]:
final_scaler = StandardScaler()

X_train_val_scaled = final_scaler.fit_transform(X_train_val)
X_test_scaled = final_scaler.transform(X_test)

final_model = IsolationForest(
    n_estimators=100,
    contamination=best_contamination,
    random_state=42
)

final_model.fit(X_train_val_scaled)
raw_test_predictions = final_model.predict(X_test_scaled)
y_test_pred = (raw_test_predictions == -1).astype(int)

print(confusion_matrix(y_test, y_test_pred, labels=[0,1]))
print(classification_report(y_test, y_test_pred, labels=[0, 1], target_names=["Normal", "Anomaly"], zero_division=0))

[[190   0]
 [  1   9]]
              precision    recall  f1-score   support

      Normal       0.99      1.00      1.00       190
     Anomaly       1.00      0.90      0.95        10

    accuracy                           0.99       200
   macro avg       1.00      0.95      0.97       200
weighted avg       1.00      0.99      0.99       200

